# RetinaScan — DR Severity Grading Training (SIH26038)
Train EfficientNetB0 5-class grader on APTOS 2019 → tune referable-DR threshold (>90% sens / >85% spec) → calibrate → ONNX export for MATLAB.

**Flow:** preprocess → split by patient → train → QWK + sens/spec → temperature scaling → Grad-CAM check → ONNX out.

In [ ]:
# ============================== SETUP ===================================
!pip -q install timm pytorch-grad-cam onnx onnxruntime scikit-learn opencv-python-headless
import os, cv2, numpy as np, pandas as pd, torch, timm, json, math
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import cohen_kappa_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(42); np.random.seed(42)
print('device:', device)

In [ ]:
# ======================== GET APTOS 2019 ================================
# Option A: upload train.zip from Kaggle and unzip here.
# Option B (drive): mount and unzip from Drive.
# Expect: ./aptos/train_images/*.png and ./aptos/train.csv (id_code,diagnosis)
APTOS_DIR = './aptos'
os.makedirs(APTOS_DIR, exist_ok=True)
df = pd.read_csv(f'{APTOS_DIR}/train.csv')
print(df.diagnosis.value_counts().sort_index())

In [ ]:
# ================= BEN-GRAHAM PREPROCESS (cache to ./prep) ==============
PREP_DIR = './prep512'; os.makedirs(PREP_DIR, exist_ok=True)
def ben_graham(path, size=512):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    m = (gray > 12).astype(np.uint8)*255
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, np.ones((15,15),np.uint8))
    ys, xs = np.where(m>0)
    if len(xs)>100:
        img = img[ys.min():ys.max()+1, xs.min():xs.max()+1]
    img = cv2.resize(img, (size,size))
    lab = cv2.cvtColor(img, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab = cv2.merge((clahe.apply(l), a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

from concurrent.futures import ThreadPoolExecutor
def cache_one(row):
    out = f"{PREP_DIR}/{row.id_code}.png"
    if not os.path.exists(out):
        cv2.imwrite(out, cv2.cvtColor(ben_graham(f"{APTOS_DIR}/train_images/{row.id_code}.png"), cv2.COLOR_RGB2BGR))
    return out
with ThreadPoolExecutor(8) as ex:
    df['path'] = list(ex.map(cache_one, [r for r in df.itertuples()]))
print('cached', len(df))

In [ ]:
# ============== SPLIT (by patient-ish: id prefix as group guard) ========
# APTOS ids are per-image; GroupShuffleSplit on id_code keeps duplicates/leaks out.
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_i, va_i = next(gss.split(df, df.diagnosis, groups=df.id_code))
train_df, val_df = df.iloc[tr_i].reset_index(drop=True), df.iloc[va_i].reset_index(drop=True)
print('train', len(train_df), 'val', len(val_df))

class FundusDS(Dataset):
    def __init__(self, d, augment=False):
        self.d, self.augment = d, augment
    def __len__(self): return len(self.d)
    def __getitem__(self, i):
        r = self.d.iloc[i]
        x = cv2.imread(r.path)[:, :, ::-1].astype(np.float32)/255.0
        if self.augment:
            if np.random.rand()<.5: x = x[:, ::-1]
            if np.random.rand()<.5: x = x[::-1]
            k = np.random.randint(0,4); x = np.rot90(x, k).copy()
            if np.random.rand()<.3:  # illumination jitter
                x = np.clip(x * (0.85 + 0.3*np.random.rand()), 0, 1)
        x = (x - np.array([.485,.456,.406]))/np.array([.229,.224,.225])
        return torch.tensor(x.transpose(2,0,1), dtype=torch.float32), int(r.diagnosis)

train_ld = DataLoader(FundusDS(train_df, True), batch_size=32, shuffle=True, num_workers=2)
val_ld   = DataLoader(FundusDS(val_df), batch_size=64, num_workers=2)

In [ ]:
# ================== MODEL: EfficientNetB0 transfer ======================
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=5).to(device)
# class weights (imbalance: class 0 dominates)
cnt = train_df.diagnosis.value_counts().sort_index().values
w = torch.tensor(cnt.sum()/ (len(cnt)*cnt), dtype=torch.float32).to(device)
crit = nn.CrossEntropyLoss(weight=w)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=8)

def qwk_val():
    model.eval(); P, T = [], []
    with torch.no_grad():
        for x, y in val_ld:
            P += model(x.to(device)).argmax(1).cpu().tolist(); T += y.tolist()
    model.train()
    return cohen_kappa_score(T, P, weights='quadratic')

EPOCHS = 8
best = -1
for e in range(EPOCHS):
    tot, corr, loss_sum = 0, 0, 0.0
    model.train()
    for x, y in train_ld:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        out = model(x)
        loss = crit(out, y)
        loss.backward(); opt.step()
        loss_sum += loss.item(); corr += (out.argmax(1)==y).sum().item(); tot += len(y)
    sched.step()
    k = qwk_val()
    print(f'ep{e+1}: loss {loss_sum/len(train_ld):.4f} acc {corr/tot:.3f} valQWK {k:.4f}')
    if k > best:
        best = k; torch.save(model.state_dict(), 'best_dr.pt')
print('best QWK:', best)

In [ ]:
# ======== REFERABLE-DR THRESHOLD TUNING (PS: sens>90%, spec>85%) ========
model.load_state_dict(torch.load('best_dr.pt')); model.eval()
probs, trues = [], []
with torch.no_grad():
    for x, y in val_ld:
        p = torch.softmax(model(x.to(device)), 1).cpu().numpy()
        probs.append(p); trues.append(y.numpy())
probs = np.concatenate(probs); trues = np.concatenate(trues)
ref_score = probs[:, 2] + probs[:, 3] + probs[:, 4]      # P(level >= 2)
ref_true = (trues >= 2).astype(int)
best_t, best_pair = 0.5, (0, 0)
for t in np.linspace(0.05, 0.9, 86):
    pred = (ref_score >= t).astype(int)
    tp = ((pred==1)&(ref_true==1)).sum(); fn = ((pred==0)&(ref_true==1)).sum()
    tn = ((pred==0)&(ref_true==0)).sum(); fp = ((pred==1)&(ref_true==0)).sum()
    sens, spec = tp/(tp+fn+1e-9), tn/(tn+fp+1e-9)
    if sens >= 0.90 and spec > best_pair[1]:
        best_t, best_pair = t, (sens, spec)
print(f'threshold {best_t:.3f} -> sens {best_pair[0]:.3f} spec {best_pair[1]:.3f}')
print('referable AUC:', roc_auc_score(ref_true, ref_score))
print(confusion_matrix(ref_true, (ref_score>=best_t).astype(int)))
json.dump({'threshold': float(best_t)}, open('referable_meta.json','w'))

In [ ]:
# ============== TEMPERATURE CALIBRATION (calibrated confidence) =========
logits_all = []
with torch.no_grad():
    for x, y in val_ld:
        logits_all.append(model(x.to(device)).cpu()); 
logits_all = torch.cat(logits_all)
T = torch.nn.Parameter(torch.ones(1)*1.5)
optT = torch.optim.LBFGS([T], lr=0.05, max_iter=60)
tlabels = torch.tensor(trues)
def closure():
    optT.zero_grad()
    l = nn.functional.cross_entropy(logits_all / T, tlabels)
    l.backward(); return l
optT.step(closure)
print('temperature:', T.item())
json.dump({'threshold': float(best_t), 'temperature': float(T.item())}, open('referable_meta.json','w'))

In [ ]:
# ===================== GRAD-CAM SANITY CHECK ============================
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
target_layers = [model.blocks[-1]]
cam = GradCAM(model=model, target_layers=target_layers)
import matplotlib.pyplot as plt
for i in range(3):
    r = val_df.iloc[i]
    img = cv2.imread(r.path)[:,:,::-1].astype(np.float32)/255.0
    xin = (img - np.array([.485,.456,.406]))/np.array([.229,.224,.225])
    tin = torch.tensor(xin.transpose(2,0,1)[None], dtype=torch.float32).to(device)
    g = cam(input_tensor=tin)[0]
    vis = show_cam_on_image(img, g, use_rgb=True)
    plt.figure(figsize=(8,4)); plt.subplot(1,2,1); plt.imshow(img); plt.title(f'true {r.diagnosis}')
    plt.subplot(1,2,2); plt.imshow(vis); plt.title('Grad-CAM'); plt.show()

In [ ]:
# =================== ONNX EXPORT FOR MATLAB =============================
dummy = torch.randn(1, 3, 512, 512, device=device)
torch.onnx.export(model, dummy, 'dr_grading.onnx',
                  input_names=['input'], output_names=['logits'],
                  dynamic_axes={'input':{0:'batch'}, 'logits':{0:'batch'}},
                  opset_version=17)
import onnxruntime as ort
sess = ort.InferenceSession('dr_grading.onnx')
o = sess.run(None, {'input': np.random.randn(1,3,512,512).astype(np.float32)})[0]
print('ONNX ok, out shape', o.shape)

# In MATLAB (Deep Learning Toolbox):
#   net = importONNXNetwork('dr_grading.onnx');
#   scores = predict(net, dlarray(single(img),'SSCB'));
#   referable = sum(softmax(scores)(3:5)) >= threshold;  % from referable_meta.json
# Also copy referable_meta.json next to the ONNX file for the app.